# Spark History Server lab (module 16) from a local notebook — Livy + sparkmagic

This notebook is the same exercise as [`lab_basics_spark_history.md`](../lab_basics_spark_history.md), for participants who work from their laptop instead of `pyspark3` on the edge node. You run **one** Spark application in nine small steps and look it up in the **Spark History Server** after every step. The lab document explains the UI pages in full; this notebook has the code and a short list of what to look at.

Each `%%spark` cell holds the code you would type at the `>>>` prompt. `sparkmagic` sends it to a **Livy session** on the cluster, where it runs in a YARN container as **you**. Your laptop needs no Spark installation and no Kerberos.

**Before you start:** set up Jupyter exactly as in [module 09, section 6](../../../09-basics-livy/lab/lab_basics_livy.md) (Python 3.12 environment, `pip install -r ../../../09-basics-livy/lab/notebooks/requirements.txt`, and the `sparkmagic` config template copied to `~/.sparkmagic/config.json`). You also need the **Livy URL** and the **Spark History Server URL** from your handout. Keep the History Server open in a browser tab next to this notebook.

**What differs from `pyspark3`**

| `pyspark3` on the edge node | This notebook |
|---|---|
| The driver runs on the edge node (client mode) | The driver runs in a **YARN container** (Livy uses cluster mode) |
| Paths and user come from shell `export`s (`os.environ`) | The cluster session does not see your shell: section 0 sets the paths in the session |
| The application name comes from `--name` | The application name is the Livy session name |
| `exit()` ends the application | `%spark delete` ends it (last cell). Until then it is an **incomplete** application in the History Server |

## How the cells work

| Cell starts with | Runs on | Meaning |
|---|---|---|
| nothing, or `%spark ...` | your **laptop** | plain Python, or session management (`add`, `info`, `delete`) |
| `%%spark -s spark_history` | the **cluster** | PySpark code in your session `spark_history`, like typing it at `>>>` (use `print`, it is always shown) |

All `%%spark` cells with the same `-s` name share one session: `spark`, variables, DataFrames and the helper functions stay alive from cell to cell. Run the cells **in order**. If you restart the kernel or delete the session, start again at the connection cells.

The cell in section 9 **fails on purpose** (a marked failure case). Run the cells one by one there, not with *Run all*.

## Connection and session

The password is asked with `getpass`: it stays in the memory of this kernel and is never written into the notebook file.

In [ ]:
import getpass

livy_url = input("Livy URL (from your handout, no trailing slash): ").strip().rstrip("/")
ws_user = input("User name (WORKSHOP_USER): ").strip()
ldap_password = getpass.getpass(f"LDAP password for {ws_user}: ")

In [ ]:
%load_ext sparkmagic.magics

`sparkmagic` sends your settings from `~/.sparkmagic/config.json` to Livy when it creates the session. The next cell adds a **session name** to them, before the session is created. This is the name you search for in the History Server (column *App Name*).

In [ ]:
from datetime import datetime
from sparkmagic.utils import configuration as conf

# Livy session name (shows up in Livy and the History Server). It must be unique on the Livy server,
# so it has your user name and the start time. The label after -s in the cells below is a different thing: it only exists in this notebook.
session_cfg = conf.session_configs()          # settings from ~/.sparkmagic/config.json
session_cfg["name"] = f"{ws_user}-spark-history-{datetime.now():%H%M%S}"
conf.override(conf.session_configs.__name__, session_cfg)
session_cfg["name"]

`%spark add` creates the Livy session. It takes about a minute: the progress bar shows `starting`, then `idle`. `-k` makes the cell safe to run twice.

In [ ]:
%spark add -s spark_history -l python -u $livy_url -a $ws_user -p $ldap_password -t Basic_Access -k

## 0. Orient — your application and the History Server

**Goal:** Have a running application, find it in the History Server, and label your code so every UI row can be traced back to it.

The session runs on the cluster and does not see your shell, so set the paths here. `WORKSHOP_ROOT` is the workshop folder of your handout (default below); the user and the home folder follow from the user the session runs as.

In [ ]:
%%spark -s spark_history

import getpass

workshop_root = "/2026_cdp_workshop"          # change only if your handout says otherwise
workshop_user = getpass.getuser()             # the session runs as you
workshop_home = "/user/" + workshop_user
meter_csv     = workshop_root + "/data/grid_measurements/sample-grid-measurements.csv"
out_path      = workshop_home + "/spark-history-out"
print(workshop_user, out_path)

Print the identity of your application. The **application id** is the key for everything in the History Server (YARN application, event log file, list row). The **event log dir** is where Spark writes one JSON line per event while the application runs.

In [ ]:
%%spark -s spark_history

sc = spark.sparkContext
print("application id  :", sc.applicationId)
print("application name:", sc.appName)
print("live UI         :", sc.uiWebUrl)
print("event log dir   :", sc.getConf().get("spark.eventLog.dir", "(not set)"))

Four settings that change what you will see in the UI. Note them; you find them again in the **Environment** tab in section 10.

In [ ]:
%%spark -s spark_history

for key in ["spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions",
            "spark.sql.autoBroadcastJoinThreshold", "spark.task.maxFailures"]:
    print(key, "=", spark.conf.get(key, "(not set, Spark default)"))

**Open the History Server** (URL from your handout, or Cloudera Manager → Spark 3 → *Spark3 History Server Web UI*).

- Your application is **not** in the default list: it is still running, and the default list shows completed applications only. Click **Show incomplete applications** and search for your user name or `spark-history`.
- Columns: **App ID**, **App Name** (the Livy session name), **Started**, **Completed** (empty while it runs), **Duration**, **Spark User**, **Event Log** (the raw JSON file; the History Server rebuilds every page from it).
- Click the **App ID**. The **Jobs** tab is still empty.
- If the application is missing: wait about 10 seconds, reload, compare the id.

> **SCREENSHOT** `images/01-history-app-list.png`, `images/02-history-show-incomplete.png`, `images/03-app-empty-jobs-tab.png` (see the lab document).

**The tabs and their questions:** *Jobs* (which action was slow or failed?), *Stages* (where does the time go?), *Storage* (what is cached?), *Environment* (which settings apply?), *Executors* (who did the work, which logs?), *SQL / DataFrame* (what did Spark run for this line of code?). The chain is always **code line → query → job → stage → task**.

**Label your code.** PySpark job descriptions look like `count at NativeMethodAccessorImpl.java:0` and never point at your line. `step()` puts a label on every job that follows; `jobs()` prints the job ids a step has started, so you can look for exactly those numbers in the UI.

In [ ]:
%%spark -s spark_history

from pyspark.sql import functions as F

# the ledger remembers what each step was about
ledger = {}

In [ ]:
%%spark -s spark_history

def step(key, text):
    # every job started after this call is labelled "<key>: <text>" in the UI
    spark.sparkContext.setJobGroup(key, f"{key}: {text}")
    ledger[key] = text

In [ ]:
%%spark -s spark_history

def jobs(key):
    # the job ids this step has started so far: compare them with the Jobs tab
    ids = sorted(spark.sparkContext.statusTracker().getJobIdsForGroup(key))
    print(f"{key} - {ledger[key]}: job ids {ids}")

## 1. A job, its stages and its tasks

**Goal:** Read one complete job: from the Jobs tab down to a single task.

Create the label, define the CSV reader with an explicit schema, and ask which jobs exist so far. The list is empty (`[]`): reading is **lazy**.

In [ ]:
%%spark -s spark_history

step("S1", "read csv with explicit schema, count")
meter_schema = (
    "ts_utc STRING, station_id STRING, device_id STRING, phase STRING, "
    "active_power_kw DOUBLE, reactive_power_kvar DOUBLE, voltage_v DOUBLE, "
    "current_a DOUBLE, power_factor DOUBLE, frequency_hz DOUBLE, "
    "transformer_load_pct DOUBLE, quality_flag STRING, event_code STRING, "
    "comm_status STRING")
meter = spark.read.option("header", True).schema(meter_schema).csv(meter_csv)
jobs("S1")

Now trigger an action. You should see **15** rows and a job id. Reload the History Server (wait about 10 seconds if the job is missing).

In [ ]:
%%spark -s spark_history

print(meter.count())
jobs("S1")

**Look at**

- **Jobs tab:** your label in **Description**, the **Job Id** equal to the printed id, **Stages** and **Tasks** (succeeded/total), the **Event Timeline** (executors are added on demand).
- **Job page:** the **DAG Visualization**: stage 1 = `FileScan csv` + partial aggregate, then an **Exchange**, stage 2 = final sum. A `count()` usually runs as **two stages**. Note that job ids and stage ids are separate counters.
- **Stage page:** one row per **task** (a task = one partition on one executor; one task for this tiny file). **Summary Metrics** show min / 25th / median / 75th / max for Duration, GC Time, Input Size / Records. About 15 input records.

> **SCREENSHOT** `images/04-jobs-tab-s1.png`, `images/05-job-detail-dag-s1.png`, `images/06-stage-detail-s1.png`

**Check question:** Why does a `count()` over 15 rows run as two stages and not one?

## 2. A job you did not ask for

**Goal:** Find a job that Spark started on its own.

Read the same file with `inferSchema`. You call no action here, yet the list of job ids is **not empty**: Spark reads the whole file once to guess the column types.

In [ ]:
%%spark -s spark_history

step("S2", "read csv with inferSchema")
meter_inferred = (spark.read.option("header", True)
    .option("inferSchema", True).csv(meter_csv))
jobs("S2")

In [ ]:
%%spark -s spark_history

meter_inferred.printSchema()
jobs("S2")

**Look at:** the `S2` row in the Jobs tab: it has **Input** but no result you ever see. In the stage page open **+details**: the call site (`csv at NativeMethodAccessorImpl.java:0`) is what *every* job looked like without your label. On a landing folder with thousands of files this is a full extra pass, which is why we use explicit schemas.

> **SCREENSHOT** `images/07-jobs-tab-s2-inferschema.png`

**Check question:** Which line started the extra job, the `.csv(...)` or `printSchema()`? How do you know from the printed ids?

## 3. Narrow transformations: one stage, and the plan

**Goal:** Connect a few lines of DataFrame code to a plan graph and to a single-stage job.

Three transformations; the printed list stays empty because nothing runs until an action.

In [ ]:
%%spark -s spark_history

step("S3", "filter, new column, show")
meter_ok = (meter
    .withColumn("ts_utc", F.to_timestamp("ts_utc"))
    .filter(F.col("quality_flag") == "OK")
    .withColumn("power_w", F.col("active_power_kw") * 1000))
jobs("S3")

Print the plan (no job), then run the action. `explain()` prints the text version of the plan; the same node names appear in the UI.

In [ ]:
%%spark -s spark_history

# the plan first (no job), then the action
meter_ok.explain()
meter_ok.select("ts_utc", "station_id", "active_power_kw", "power_w").show(5, False)
jobs("S3")

**Look at**

- **SQL / DataFrame tab:** one row per query with **Duration** and **Job IDs** (links): the bridge from a query to its jobs.
- Click the query: the **plan graph** reads bottom to top, `Scan csv` → `Filter` → `Project` → `CollectLimit`, the nodes of the `explain()` output with real numbers (**number of output rows**). Click **Details** for the plan text, including `PushedFilters` and `ReadSchema`.
- **Jobs tab:** the job has **one stage** and no `Exchange`: `filter`, `withColumn` and `select` are *narrow* transformations and never start a new stage.

> **SCREENSHOT** `images/08-sql-tab-s3-plan.png`

**Check question:** `meter_ok` was defined before `explain()`. Why is there no job for the `filter`?

## 4. Tasks follow partitions

**Goal:** Create data that is big enough to give the metrics meaning, and see that tasks = partitions.

The sample CSVs have 15 rows. From here on you generate **5 million readings** for three stations in Spark: `TS-101` gets 70 % of the rows, `TS-204` 20 %, `TS-305` 10 %, timestamps between 08:00 and 12:00 on 2026-01-15, split into **8 partitions**.

In [ ]:
%%spark -s spark_history

step("S4", "build readings_big: 5 million generated rows, 8 partitions")
n_rows = 5_000_000
base_ts = F.lit("2026-01-15 08:00:00").cast("timestamp").cast("long")
readings_big = (spark.range(0, n_rows, 1, 8)
    .withColumn("station_id", F.when(F.col("id") % 10 < 7, "TS-101")
                               .when(F.col("id") % 10 < 9, "TS-204")
                               .otherwise("TS-305"))
    .withColumn("ts_utc", (base_ts + F.col("id") % 14400).cast("timestamp"))
    .withColumn("active_power_kw", 150 + (F.col("id") % 97) * 0.9)
    .withColumn("quality_flag", F.when(F.col("id") % 50 == 0, "WARN").otherwise("OK")))
print("partitions:", readings_big.rdd.getNumPartitions())

You should see `partitions: 8`. Now count (expect **5000000**).

In [ ]:
%%spark -s spark_history

print(readings_big.count())
jobs("S4")

**Look at:** the first stage of the S4 job has **8 tasks**, one per partition (you chose `8` in `spark.range(0, n_rows, 1, 8)`). In **Summary Metrics** min, median and max **Duration** are close: this is what *healthy* looks like. **Input** is empty (generated data). The second stage has one task (it sums eight partial counts).

> **SCREENSHOT** `images/09-stage-tasks-s4.png`

**Check question:** How many tasks would the first stage have with `spark.range(0, n_rows, 1, 16)`? What would change in the Summary Metrics?

## 5. A shuffle splits the job

**Goal:** Find the stage boundary created by `groupBy` and read the shuffle columns.

Average power and number of readings per station and hour. In the plan find `HashAggregate` twice with an `Exchange hashpartitioning(...)` in between: **every `Exchange` is a stage boundary**.

In [ ]:
%%spark -s spark_history

step("S5", "shuffle: average power per station and hour")
hourly = (readings_big
    .withColumn("hour_utc", F.date_trunc("hour", F.col("ts_utc")))
    .groupBy("station_id", "hour_utc")
    .agg(F.avg("active_power_kw").alias("avg_power_kw"),
         F.count(F.lit(1)).alias("n")))
hourly.explain()

Run it (12 result rows: 3 stations x 4 hours).

In [ ]:
%%spark -s spark_history

rows = hourly.collect()
print(len(rows), "result rows")
jobs("S5")

**Look at (job page and Stages tab)**

- The DAG has (at least) **two stages** around the `Exchange`. Stage 1 has **8 tasks**.
- **Shuffle Write** (stage 1) and **Shuffle Read** (stage 2): stage 1 processed 5,000,000 rows but writes only a few dozen records. The first `HashAggregate` is the *partial* aggregate, so only partial results are shuffled.
- The `Exchange` is planned for **200** partitions (`spark.sql.shuffle.partitions`), yet the next stage has only a few tasks: **adaptive query execution** merges tiny partitions (look for `AQEShuffleRead` / *coalesced* in the SQL tab). With adaptive execution off you see 200 tasks, most with 0 records. You may also see **skipped stages**.
- **Stages tab:** sort by **Duration** to find the slowest stage of an application.

> **SCREENSHOT** `images/10-job-dag-s5.png`

**Check question:** The first stage reads 5 million rows. Why is its Shuffle Write only a few dozen records?

## 6. Broadcast join or sort-merge join

**Goal:** Compare two joins with the same answer but different plans, and recognize **skew**.

First the default: the tiny station table is built by hand and *broadcast* to every executor. Find `BroadcastExchange` and `BroadcastHashJoin` in the plan; the only shuffle is the `groupBy` at the end.

In [ ]:
%%spark -s spark_history

step("S6a", "join with station_ref, broadcast")
station_ref = spark.createDataFrame(
    [("TS-101", "Nord"), ("TS-204", "Sued"), ("TS-305", "West")],
    ["station_id", "region"])
q_broadcast = (readings_big.join(station_ref, "station_id")
    .groupBy("region").count())
q_broadcast.explain()

Run it. Expect 3.5 million for `Nord` (`TS-101`), 1.0 million for `Sued`, 0.5 million for `West`.

In [ ]:
%%spark -s spark_history

print(q_broadcast.collect())
jobs("S6a")

Now the same question with the `merge` hint, which forces a sort-merge join. Find `SortMergeJoin`, two `Sort` nodes and **two** `Exchange hashpartitioning(station_id, ...)`.

In [ ]:
%%spark -s spark_history

step("S6b", "join with station_ref, sort-merge (hint)")
q_merge = (readings_big.join(station_ref.hint("merge"), "station_id")
    .groupBy("region").count())
q_merge.explain()

In [ ]:
%%spark -s spark_history

print(q_merge.collect())
jobs("S6b")

**Look at**

- **SQL / DataFrame tab:** open both queries. The sort-merge exchange of the big side shows **shuffle records written** of about 5,000,000: *every row* is shuffled. In the Jobs tab compare stages and **Shuffle Write** of `S6a` and `S6b` (tens of MB against a few KB in section 5). Jobs labelled `broadcast exchange` are started by Spark itself and are not in your ledger.
- **Skew:** open the stage after the shuffle in the sort-merge job. Only three station keys exist, so at most three tasks get data, and unequal shares: `TS-101` has 70 %. In **Summary Metrics** compare **max** and **median** of **Shuffle Read Size / Records** and **Duration**; sort the task table by **Duration**. A max many times the median is the signature of **data skew**: the stage is as slow as its slowest task.

> **SCREENSHOT** `images/11-sql-plan-broadcast.png`, `images/12-sql-plan-sortmerge.png`, `images/13-stage-skew-s6b.png`

**Check question:** The broadcast variant needs no shuffle for the join. Why can it not be used when *both* tables are large?

## 7. Cache and the Storage tab

**Goal:** Find a cached DataFrame and see how the second action gets cheaper.

`cache()` is lazy too: nothing is stored until an action reads the data.

In [ ]:
%%spark -s spark_history

step("S7", "cache readings_ok, two actions")
readings_ok = readings_big.filter(F.col("quality_flag") == "OK").cache()
jobs("S7")

The first action fills the cache, the second one reads from it (about 4.9 million readings, three station counts, two or more job ids).

In [ ]:
%%spark -s spark_history

# the first action fills the cache, the second one reads from it
print(readings_ok.count())
print(readings_ok.groupBy("station_id").count().collect())
jobs("S7")

**Look at**

- **Storage tab:** one entry for `readings_ok` (named after its plan). **Cached Partitions** 8, **Fraction Cached** should be 100 %, **Size in Memory** shows the columnar cache size. A fraction below 100 % or **Size on Disk** above 0 means the executors lacked memory. Click the entry to see which executor holds which partition.
- **SQL / DataFrame tab:** the second query has an `InMemoryTableScan` node instead of the range generator, and its first stage is much shorter (compare durations in the Jobs tab).

> **SCREENSHOT** `images/14-storage-tab.png`

Release the memory and reload the Storage tab: the entry disappears.

In [ ]:
%%spark -s spark_history

_ = readings_ok.unpersist()

**Check question:** If you had skipped `cache()`, what would the second action have done?

## 8. Write Parquet: tasks, files and output metrics

**Goal:** Relate the output columns of a write stage to the files on HDFS and to partition pruning on the read side.

Write the readings as Parquet, one folder per station. The cache was released, so Spark computes the data again.

In [ ]:
%%spark -s spark_history

step("S8", "write parquet partitioned by station_id, read back")
readings_ok.write.mode("overwrite").partitionBy("station_id").parquet(out_path)
jobs("S8")

**Look at:** the write stage has **8 tasks** and an **Output Size / Records** column (about 4.9 million records in total). In the **SQL / DataFrame** tab the top node `InsertIntoHadoopFsRelationCommand` has **number of written files** and **written output**.

> **SCREENSHOT** `images/15-sql-write-metrics.png`

Compare with the files. From a terminal on the edge node (valid ticket), or from the HDFS UI of your handout; replace `<you>` by your user name:

```bash
hdfs dfs -ls /user/<you>/spark-history-out
hdfs dfs -ls /user/<you>/spark-history-out/station_id=TS-305
```

You should see one `station_id=...` directory per station and several part files in each. Now read the data back and ask for one station only.

In [ ]:
%%spark -s spark_history

# read it back: a different scan node, and only one station directory is needed
back = spark.read.parquet(out_path)
one_station = back.filter(F.col("station_id") == "TS-305")
one_station.explain()
print(one_station.count())
jobs("S8")

**Look at:** you should see about 0.5 million rows. In the plan find `FileScan parquet` with `PartitionFilters: [..., (station_id = TS-305)]`: Spark skips the other station folders. In the SQL tab compare **number of files read** and **size of files read** with the number of files you listed.

**Check question:** The stage has 8 tasks and there are 3 stations. How many part files does Spark write, and why?

## 9. A failing job

**Goal:** Read a failure in the History Server: failed job, failed stage, task attempts, error and logs.

> 🔴 **Failure case — an error is expected here.** A Python function raises an error on one specific row. Run the next two cells, read the message, then find the failure in the UI. Your session stays alive.

Define the function and the label first. It fails for the id `4242`, which lives in the first of four partitions.

In [ ]:
%%spark -s spark_history

step("S9", "FAILURE CASE: python udf raises on one row")

def risky(x):
    # one bad row, like a corrupt reading
    if x == 4242:
        raise ValueError(f"bad reading with id {x}")
    return float(x)

risky_udf = F.udf(risky, "double")

Run the failing action. The cell turns red with `Job aborted due to stage failure: Task 0 in stage ... failed 4 times` and `ValueError: bad reading with id 4242`.

In [ ]:
%%spark -s spark_history

# FAILURE CASE: expected to fail with "Job aborted due to stage failure ... bad reading with id 4242"
(spark.range(0, 100000, 1, 4)
    .select(risky_udf("id").alias("v"))
    .agg(F.sum("v"))
    .collect())

In [ ]:
%%spark -s spark_history

jobs("S9")

**Look at**

- **Jobs tab:** the job is under **Failed Jobs**. Click it: **Failed Stages** holds the stage. Click the stage.
- **Task table:** task index **0** appears **four times** with **Attempt** 0, 1, 2, 3, all `FAILED`: Spark retries up to `spark.task.maxFailures` (the value you printed in section 0), then aborts the job. The other tasks are `SUCCESS` or `KILLED`.
- **Errors column:** `PythonException` with the Python traceback ending in your `ValueError`; the traceback names your function `risky`.
- **Executor ID / Logs:** the **stderr** / **stdout** links open the container log; after the application ended the same logs come from `yarn logs -applicationId <application-id>`.
- **Stages tab** (Failed Stages) and **SQL / DataFrame tab** (status `FAILED`) show the same failure.

> **SCREENSHOT** `images/16-jobs-tab-failed.png`, `images/17-failed-task-attempts.png`

Confirm the session survived:

In [ ]:
%%spark -s spark_history

print(spark.range(10).count())

**Check question:** Why did Spark run the failing task four times instead of failing at the first error? Would a retry help if the data itself is bad?

## 10. End the application and read it as completed

**Goal:** See the difference between a running (incomplete) and a finished application, and read the application-wide tabs.

Print the ledger first: every step with its job ids, in the order of the Jobs tab. Compare it with the Jobs tab of your application.

In [ ]:
%%spark -s spark_history

for key in ledger:
    jobs(key)
print("application id:", sc.applicationId)

Closing the notebook does **not** stop the Livy session — the Spark application keeps running on YARN until Livy's idle timeout, and it stays *incomplete* in the History Server. `%spark delete` ends it. Run it every time.

In [ ]:
%spark delete -s spark_history

Reload the History Server and open the **default** list: your application is there now, with a **Completed** time and a **Duration**.

> **SCREENSHOT** `images/18-history-completed.png`

Go through the application-wide tabs:

- **Jobs:** the event timeline ends with executors *removed*. Which step was the most expensive? Compare duration and tasks.
- **Executors:** the driver and all executors, **dead** now. Per executor: **Task Time (GC Time)** (red above about 10 %: memory shortage), **Input**, **Shuffle Read / Write**, **stdout / stderr** links. `images/19-executors-tab.png`
- **Environment:** search `spark.eventLog`, `spark.sql.adaptive`, `spark.sql.shuffle.partitions`, `spark.dynamicAllocation` and compare with section 0. `images/20-environment-tab.png`
- **Storage:** empty again: the cache dies with the application, as do your variables.

Fill in this table from the UI (one example is given) and compare with a neighbour:

| Step | Job ids (from the ledger) | Stages | Tasks | The number that tells the story |
|------|---------------------------|--------|-------|---------------------------------|
| S1 count | | 2 (typically) | | Input records |
| S5 shuffle | | | | Shuffle Write records |
| S6b sort-merge | | | | max vs median Shuffle Read |
| S7 cache | | | | Fraction Cached |
| S9 failure | | | | Attempt 0–3 |

**Check question:** A colleague sends you only an application id for a job that ran last night. List the pages you would open, in order, to find out whether it was slow, and why.

For the stretch exercises see [`lab_basics_spark_history.md`](../lab_basics_spark_history.md).